In [15]:
from langchain_ollama import ChatOllama
from typing import TypedDict 
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

In [16]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [17]:
llm = ChatOllama(model="mistral", temperature=0.7)

def generate_joke(state: JokeState) -> JokeState:
    topic = state["topic"]
    prompt = f"You are a funny comedian. Tell me a joke about {topic}."
    joke = llm.invoke(prompt).content
    return {"joke" : joke}

def explain_joke(state: JokeState) -> JokeState:
    joke = state["joke"]
    prompt = f"Explain the following joke in simple terms: {joke}"
    explanation = llm.invoke(prompt).content
    return {"explanation": explanation}

In [18]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

checkpointer = InMemorySaver()

model = graph.compile(checkpointer=checkpointer)

In [19]:
config1 = {
    "configurable" : {"thread_id" : '1'}
}
query1 = {"topic": "programming"}
# output = model.invoke(query1, config=config1)
# we are going to use the model.stream method to stream the output of the model in real-time
for message_chunk, metadata in model.stream(query1, config=config1, stream_mode="messages"):
    if message_chunk.content:
        print(message_chunk.content, end=" ", flush=True)

 Why  did  the  program mer  quit  his  job  at  the  b ak ery ? 
 
 Because  he  didn ' t  like  the  software  in  the  oven ! 
 
 ( I  hope  this  brings  a  smile  to  your  face !)  This  joke  is  a  play  on  words ,  combining  the  worlds  of  programming  ( comput ers )  and  baking  ( cook ing ). 
 
 In  simple  terms ,  the  punch line  " Because  he  didn ' t  like  the  software  in  the  oven "  means  that  the  program mer  was  unhappy  with  the  equipment  or  tools  used  for  baking  in  the  b ak ery .  In  programming ,  " soft ware "  refers  to  computer  programs .  So ,  it ' s  a  hum orous  way  of  saying  that  the  program mer  didn ' t  like  the  o vens  ( or  perhaps  other  equipment )  used  for  baking . 
 
 The  joke  is  am using  because  it  takes  something  typically  associated  with  computers  and  applies  it  in  an  unexpected  context ,  creating  a  surprise  or  humor . 

In [20]:
output['joke']

' Why did the Java developer buy a copy of "Catch-22"?\n\nBecause he wanted to read something about exceptions!\n\n(I\'ll be here all week, folks!)'

In [21]:
model.get_state(config=config1)

StateSnapshot(values={'topic': 'programming', 'joke': " Why did the programmer quit his job at the bakery?\n\nBecause he didn't like the software in the oven!\n\n(I hope this brings a smile to your face!)", 'explanation': ' This joke is a play on words, combining the worlds of programming (computers) and baking (cooking).\n\nIn simple terms, the punchline "Because he didn\'t like the software in the oven" means that the programmer was unhappy with the equipment or tools used for baking in the bakery. In programming, "software" refers to computer programs. So, it\'s a humorous way of saying that the programmer didn\'t like the ovens (or perhaps other equipment) used for baking.\n\nThe joke is amusing because it takes something typically associated with computers and applies it in an unexpected context, creating a surprise or humor.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15dc7a-f169-6b24-8002-1c54bd8b0793'}}, metadata={'source': 'l

In [22]:
list(model.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'programming', 'joke': " Why did the programmer quit his job at the bakery?\n\nBecause he didn't like the software in the oven!\n\n(I hope this brings a smile to your face!)", 'explanation': ' This joke is a play on words, combining the worlds of programming (computers) and baking (cooking).\n\nIn simple terms, the punchline "Because he didn\'t like the software in the oven" means that the programmer was unhappy with the equipment or tools used for baking in the bakery. In programming, "software" refers to computer programs. So, it\'s a humorous way of saying that the programmer didn\'t like the ovens (or perhaps other equipment) used for baking.\n\nThe joke is amusing because it takes something typically associated with computers and applies it in an unexpected context, creating a surprise or humor.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15dc7a-f169-6b24-8002-1c54bd8b0793'}}, metadata={'source': '